# Network Analysis

Builds a similarity graph over BERTopic subtopics across all four mental-health categories, then tests whether that graph shows real category-specific structure (vs. what random chance would produce).

## 0. Custom topic names

Human-readable names assigned to each BERTopic topic ID, per category (from manual inspection of topic keywords in notebook 1).

In [ ]:
custom_names_dict = {
    "Anxiety": {
        -1: "Stress Sleep", 0: "Community Support", 1: "Home Relations", 2: "Work Stress",
        3: "Panic Symptoms", 4: "ADHD Medication", 5: "Family Issues", 6: "Uncertainty Skepticism",
        7: "Friendship Struggles", 8: "Medication Effects", 9: "Withdrawal Challenges",
        10: "Sleep Coping", 11: "Cognitive Symptoms",
    },
    "Depression": {
        -1: "Past Losses", 0: "Community Sharing", 1: "Family Drains", 2: "Low Energy",
        3: "Bipolar Medication", 4: "Anxiety Sadness", 5: "Financial Stress",
    },
    "PTSD and trauma": {
        -1: "Childhood Abuse", 0: "Support Skepticism", 1: "Family Abuse", 2: "Therapy Healing Fears",
        3: "Abusive Hurt Protection", 4: "Childhood Anger Trust", 5: "Financial Strain",
        6: "Pain Flashback Anger", 7: "Masked Trauma", 8: "Sexual Abuse", 9: "Family Hurt",
    },
    "Suicidal thoughts and self-harm": {
        -1: "Suicidal Pain Advice", 0: "Self-Harm Support", 1: "Suicidal Loss",
        2: "Hesitant Pain Skepticism", 3: "Family Hurt Trust", 4: "Safety Plan Trusts",
        5: "Inner Struggles", 6: "Parental Conflict", 7: "OCD Night Safety",
        8: "Financial Stress", 9: "Workplace Strain", 10: "Lonely Family", 11: "Hopelessness",
    },
}

platform = "beyondblue"
labels = ["Anxiety", "Depression", "PTSD and trauma", "Suicidal thoughts and self-harm"]


## 1. Build (or load) the subtopic graph

Run **either** 1a or 1b, not both:
- **1a** rebuilds the graph from scratch from the saved BERTopic models (slower, needed the first time or after re-running notebook 1).
- **1b** loads a previously-built graph straight from disk (fast, for re-running the analysis cells below without rebuilding).

### 1a. Rebuild from BERTopic models

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic
import networkx as nx
from collections import defaultdict
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
import pickle


def load_model_and_subtopics(platform, label, custom_names):
    """Load BERTopic model and subtopic info with topic embeddings."""
    path = os.path.join(os.getcwd(), 'data', f'{platform}_data', 'berttopic_label', label)
    try:
        topic_model = BERTopic.load(os.path.join(path, f'{label}_berttopic'))
    except Exception as e:
        print(f"Error loading model for label '{label}': {e}")
        return []

    topic_info = topic_model.get_topic_info()
    topic_keywords = {t: topic_model.get_topic(t) for t in topic_info.Topic if t != -1}
    doc_counts = topic_info.set_index("Topic")["Count"].to_dict()
    # Total number of documents assigned to this label (excluding outliers)
    total_label_docs = sum(count for tid, count in doc_counts.items() if tid != -1)

    # Get topic embeddings (excluding -1 if present)
    topic_embeddings = topic_model.topic_embeddings_
    topic_ids = sorted(topic_keywords.keys())  # Only valid topics
    if len(topic_embeddings) != len(topic_info):
        print(f"Warning: Mismatch between topic embeddings ({len(topic_embeddings)}) and topics ({len(topic_info)}) for label '{label}'.")

    subtopics = []
    for topic_id in topic_ids:
        keywords = [(kw, score) for kw, score in topic_keywords[topic_id]]
        doc_indices = [i for i, t in enumerate(topic_model.topics_) if t == topic_id]
        if not doc_indices:
            print(f"Warning: No documents assigned to topic {topic_id} in label '{label}'.")
            continue
        # Map topic_id to embedding index (assuming embeddings align with topic IDs)
        embedding_idx = topic_id + 1 if topic_id >= 0 else 0  # Adjust for -1 if present
        if embedding_idx >= len(topic_embeddings):
            print(f"Warning: No embedding for topic {topic_id} in label '{label}'.")
            continue
        subtopics.append({
            'model_label': label,
            'topic_id': topic_id,
            'name': custom_names.get(topic_id, f"{label}_{topic_id}"),
            'keywords': keywords,
            'doc_indices': doc_indices,
            'doc_count': doc_counts.get(topic_id, 0),
            'embedding': topic_embeddings[embedding_idx],
            'total_label_docs': total_label_docs,
        })

    return subtopics


def build_full_graph(platform, labels, custom_names_dict):
    all_subtopics = []

    for label in labels:
        custom_names = custom_names_dict.get(label, {})
        subtopics = load_model_and_subtopics(platform, label, custom_names)
        if not subtopics:
            print(f"Skipping label '{label}' due to loading issues.")
            continue
        all_subtopics.extend(subtopics)

    if not all_subtopics:
        print("Error: No subtopics available for graph building.")
        return nx.Graph(), None, None, None, []

    embeddings = np.array([t['embedding'] for t in all_subtopics])
    sim_matrix = cosine_similarity(embeddings)

    # Proxy corpus + dictionary for pairwise coherence between subtopics
    all_keywords = [[kw for kw, _ in t['keywords'][:10]] for t in all_subtopics]
    dictionary = Dictionary(all_keywords)

    def paired_coherence(topic1_keywords, topic2_keywords, top_n=10):
        combined = list(set(
            [kw for kw, _ in topic1_keywords[:top_n]] + [kw for kw, _ in topic2_keywords[:top_n]]
        ))
        if len(combined) < 2:
            return 0.0
        cm = CoherenceModel(topics=[combined], texts=all_keywords, dictionary=dictionary, coherence='c_v')
        return cm.get_coherence()

    G = nx.Graph()

    # Add nodes
    for idx, topic in enumerate(all_subtopics):
        relative_size = topic['doc_count'] / topic['total_label_docs']
        G.add_node(
            idx,
            label=topic['model_label'],
            name=topic['name'],
            keywords=', '.join(kw for kw, _ in topic['keywords'][:10]),
            size=relative_size,
            true_doc_count=topic['doc_count'],
        )

    n = len(all_subtopics)
    coherence_matrix = np.zeros((n, n))
    sim_values, coherence_values = [], []

    # Pass 1: add edges with weight (coherence * sim) and original_sim; store coherence
    for i in range(n):
        for j in range(i + 1, n):
            sim = sim_matrix[i, j]
            coherence = paired_coherence(all_subtopics[i]['keywords'], all_subtopics[j]['keywords'])
            coherence_matrix[i, j] = coherence
            coherence_matrix[j, i] = coherence
            sim_values.append(sim)
            coherence_values.append(coherence)
            weight = coherence * sim
            G.add_edge(i, j, weight=weight, original_sim=sim)

    sim_min = min(sim_values) if sim_values else 0.0
    sim_max = max(sim_values) if sim_values else 1.0
    coh_min = min(coherence_values) if coherence_values else 0.0
    coh_max = max(coherence_values) if coherence_values else 1.0

    print(f"Similarity range: min = {sim_min:.4f}, max = {sim_max:.4f}")
    print(f"Coherence range: min = {coh_min:.4f}, max = {coh_max:.4f}")

    # Pass 2: add nor_weight (normalized coherence * normalized sim)
    for i in range(n):
        for j in range(i + 1, n):
            sim = sim_matrix[i, j]
            coherence = coherence_matrix[i, j]
            sim_norm = (sim - sim_min) / (sim_max - sim_min) if sim_max != sim_min else 1.0
            coherence_norm = (coherence - coh_min) / (coh_max - coh_min) if coh_max != coh_min else 1.0
            G[i][j]['nor_weight'] = sim_norm * coherence_norm

    return G, sim_matrix, coherence_matrix, embeddings, all_subtopics


G, sim_matrix, coherence_matrix, embeddings, all_subtopics = build_full_graph(platform, labels, custom_names_dict)

os.makedirs('./data/network_graph', exist_ok=True)
with open('./data/network_graph/G.gpickle', 'wb') as f:
    pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
np.save('./data/network_graph/sim_matrix.npy', sim_matrix)
np.save('./data/network_graph/embeddings.npy', embeddings)
np.save('./data/network_graph/coh_matrix.npy', coherence_matrix)
with open('./data/network_graph/all_subtopics.pkl', 'wb') as f:
    pickle.dump(all_subtopics, f)


### 1b. Or: load a previously-built graph

In [ ]:
import pickle
import os
import numpy as np

data_dir = './data/network_graph'
with open(os.path.join(data_dir, 'G.gpickle'), 'rb') as f:
    G = pickle.load(f)
sim_matrix = np.load(os.path.join(data_dir, 'sim_matrix.npy'))
coherence_matrix = np.load(os.path.join(data_dir, 'coh_matrix.npy'))
embeddings = np.load(os.path.join(data_dir, 'embeddings.npy'))
with open(os.path.join(data_dir, 'all_subtopics.pkl'), 'rb') as f:
    all_subtopics = pickle.load(f)


## 2. Visualize the network

Draws subtopics as nodes (grouped/colored by category), with edges above a similarity threshold. Node position uses a per-category circular layout with manual spacing so categories don't overlap.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
import matplotlib.patches as mpatches
from sklearn.preprocessing import MinMaxScaler
import networkx as nx
import numpy as np


def plot_graph(G, labels, weight_type, threshold=0.9, pos=None):
    filtered_G = nx.Graph()

    for n, data in G.nodes(data=True):
        filtered_G.add_node(n, **data)

    cmap = get_cmap("tab10", len(labels))
    label_color = {label: cmap(i) for i, label in enumerate(labels)}
    for n in filtered_G.nodes:
        filtered_G.nodes[n]['color'] = label_color[filtered_G.nodes[n]['label']]

    filtered_edges = [(u, v, d) for u, v, d in G.edges(data=True) if d.get(weight_type, 0) >= threshold]
    for u, v, d in filtered_edges:
        filtered_G.add_edge(u, v, **d)

    if pos is None:
        # Per-category circular layout, spaced out horizontally
        pos = {}
        horizontal_scale = 1.5
        vertical_scale = 2.5
        np.random.seed(42)
        for i, label in enumerate(labels):
            nodes = [n for n, d in filtered_G.nodes(data=True) if d["label"] == label]
            subgraph = filtered_G.subgraph(nodes)
            offset_step = 12.0 * (len(nodes) / 10.0)
            sub_pos = nx.circular_layout(subgraph, scale=1.5)
            for k in sub_pos:
                sub_pos[k][0] *= horizontal_scale
                sub_pos[k][1] *= vertical_scale
                sub_pos[k][0] += i * offset_step
            pos.update(sub_pos)

        def enforce_minimum_axis_distances(pos, min_dx=3.0, min_dy=3.0, max_iter=100):
            """Ensure nodes are at least min_dx/min_dy apart, so labels don't overlap."""
            for _ in range(max_iter):
                moved = False
                nodes = list(pos.keys())
                for i in range(len(nodes)):
                    for j in range(i + 1, len(nodes)):
                        node_i, node_j = nodes[i], nodes[j]
                        xi, yi = pos[node_i]
                        xj, yj = pos[node_j]
                        dx, dy = abs(xi - xj), abs(yi - yj)

                        if dx < min_dx:
                            shift_x = (min_dx - dx) / 2
                            if xi > xj:
                                pos[node_i] = (xi + shift_x, yi)
                                pos[node_j] = (xj - shift_x, yj)
                            else:
                                pos[node_i] = (xi - shift_x, yi)
                                pos[node_j] = (xj + shift_x, yj)
                            moved = True

                        if dy < min_dy:
                            shift_y = (min_dy - dy) / 2
                            if yi > yj:
                                pos[node_i] = (xi, yi + shift_y)
                                pos[node_j] = (xj, yj - shift_y)
                            else:
                                pos[node_i] = (xi, yi - shift_y)
                                pos[node_j] = (xj, yj + shift_y)
                            moved = True

                if not moved:
                    break
            return pos

        pos = enforce_minimum_axis_distances(pos, min_dx=3.0, min_dy=4.5)

    plt.figure(figsize=(40, 25))

    node_sizes = [filtered_G.nodes[n]['size'] * 15000 for n in filtered_G.nodes]
    node_colors = [filtered_G.nodes[n]['color'] for n in filtered_G.nodes]

    if filtered_G.edges:
        edge_weights = [filtered_G[u][v][weight_type] for u, v in filtered_G.edges]
        scaler = MinMaxScaler(feature_range=(1, 15))
        scaled_weights = scaler.fit_transform(np.array(edge_weights).reshape(-1, 1)).flatten()
        # Exponential scaling emphasizes differences between similarity levels
        scaled_weights = [2.5 * (np.exp(w / 10) - 1) for w in scaled_weights]
        scaled_weights = [min(max(w, 1.0), 12.0) for w in scaled_weights]
        edge_alphas = [
            0.3 + 0.5 * ((w - min(edge_weights)) / (max(edge_weights) - min(edge_weights) + 1e-6))
            for w in edge_weights
        ]
    else:
        scaled_weights, edge_alphas = [], []

    nx.draw_networkx_nodes(filtered_G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.85)
    for (u, v), width, alpha in zip(filtered_G.edges, scaled_weights, edge_alphas):
        nx.draw_networkx_edges(filtered_G, pos, edgelist=[(u, v)], width=width, alpha=alpha)

    for node, (x, y) in pos.items():
        plt.text(
            x, y + 0.4,
            filtered_G.nodes[node]['name'],
            fontsize=20, ha='center', va='bottom', color='black',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.4),
        )

    legend_handles = [mpatches.Patch(color=label_color[label], label=label) for label in labels]
    plt.legend(handles=legend_handles, loc="upper right", fontsize=25, title="Main Topics", title_fontsize=25)

    plt.axis("off")
    plt.tight_layout()
    plt.savefig("./pic/network.svg", bbox_inches="tight")
    plt.show()

    return filtered_G, pos


norweight_threshold = 0.2
sim_threshold = 0.9946
filtered_G, pos = plot_graph(G, labels, threshold=sim_threshold, weight_type="original_sim")


## 3. Intra- vs. inter-category similarity distributions

Histograms of edge similarity, split by whether the edge connects two subtopics within the same category (intra) or across categories (inter).

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
import networkx as nx

weight_type = 'original_sim'  # one of: 'weight', 'nor_weight', 'original_sim'

categories = list({G.nodes[idx]['label'] for idx in G.nodes()})

intra_sims = defaultdict(list)
inter_sims = defaultdict(list)

for u, v, data in G.edges(data=True):
    label_u = G.nodes[u]['label']
    label_v = G.nodes[v]['label']
    value = data[weight_type]
    if label_u == label_v:
        intra_sims[label_u].append(value)
    else:
        inter_sims[label_u].append(value)
        inter_sims[label_v].append(value)

colors = {'Anxiety': 'blue', 'Depression': 'green', 'PTSD and trauma': 'red', 'Suicidal thoughts and self-harm': 'purple'}
categories = [cat for cat in ["Anxiety", "Depression", "PTSD and trauma", "Suicidal thoughts and self-harm"] if cat in categories]

for category in categories:
    if category in intra_sims and intra_sims[category]:
        plt.figure(figsize=(8, 5))
        plt.hist(intra_sims[category], bins=20, alpha=0.7, color=colors.get(category, 'gray'))
        plt.title(f"Intra-group {weight_type.capitalize()} Distribution for {category}")
        plt.xlabel(weight_type.capitalize())
        plt.ylabel("Count")
        plt.show()

    if category in inter_sims and inter_sims[category]:
        plt.figure(figsize=(8, 5))
        plt.hist(inter_sims[category], bins=20, alpha=0.7, color=colors.get(category, 'gray'))
        plt.title(f"Inter-group {weight_type.capitalize()} Distribution for {category}")
        plt.xlabel(weight_type.capitalize())
        plt.ylabel("Count")
        plt.show()


## 4. Statistical test: intra vs. inter similarity

Mann-Whitney U test (does intra-category similarity tend to be higher than inter-category?), plus a manual U-statistic/Z-score derivation and a label-shuffling permutation test as a robustness check.

In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu
from sklearn.utils import shuffle

weight_type = 'original_sim'

nodes = sorted(G.nodes())
node_labels = [G.nodes[node]['label'] for node in nodes]

intra_sims, inter_sims = [], []
for i in range(len(node_labels)):
    for j in range(i + 1, len(node_labels)):
        value = G[i][j][weight_type]
        if node_labels[i] == node_labels[j]:
            intra_sims.append(value)
        else:
            inter_sims.append(value)

print(f"Average Intra-Category {weight_type.capitalize()}: {np.mean(intra_sims):.4f}")
print(f"Average Inter-Category {weight_type.capitalize()}: {np.mean(inter_sims):.4f}")

stat, p_value = mannwhitneyu(intra_sims, inter_sims, alternative='greater')
print(f"Mann-Whitney U Test: Statistic={stat:.2f}, p-value={p_value:.4f}")
if p_value < 0.05:
    print("Significant: Intra similarities are higher than inter, suggesting category-specific structure.")

print("=" * 79)

from scipy.stats import rankdata, norm

all_sims = np.concatenate([intra_sims, inter_sims])
ranks = rankdata(all_sims)  # Average ranks for ties

ranks_intra = ranks[:len(intra_sims)]
ranks_inter = ranks[len(intra_sims):]

rank_sum_intra = np.sum(ranks_intra)
rank_sum_inter = np.sum(ranks_inter)

n1, n2 = len(intra_sims), len(inter_sims)
N = n1 + n2

U_intra_manual = rank_sum_intra - (n1 * (n1 + 1) / 2)
U_inter_manual = rank_sum_inter - (n2 * (n2 + 1) / 2)

print(f"Manual U_intra: {U_intra_manual}")
print(f"Manual U_inter: {U_inter_manual}")

U = min(U_intra_manual, U_inter_manual)
exp_U = (n1 * n2) / 2

unique, counts = np.unique(all_sims, return_counts=True)
tie_sum = np.sum(counts ** 3 - counts)
correction = (N ** 3 - N - tie_sum) / (N * (N - 1)) if tie_sum > 0 else N + 1

var_U = (n1 * n2 / 12) * correction
std_U = np.sqrt(var_U)

# Z-score with continuity correction (alternative='greater', since U_intra > exp_U)
Z = (U_intra_manual - exp_U - 0.5) / std_U
P = 1 - norm.cdf(Z)

print(f"Rank_sum_intra: {rank_sum_intra}")
print(f"number_of_intra: {n1}")
print(f"U_intra: {U_intra_manual}")
print(f"Rank_sum_inter: {rank_sum_inter}")
print(f"number_of_inter: {n2}")
print(f"U_inter: {U_inter_manual}")
print(f"U: {U}")
print(f"expected(U): {exp_U}")
print(f"standard(U): {std_U}")
print(f"Z: {Z}")
print(f"P: {P}")

print("\n" + "=" * 79 + "\n")

# Permutation test for robustness (null: shuffle category labels, 2000 times)
observed_diff = np.mean(intra_sims) - np.mean(inter_sims)
perm_diffs = []
for _ in range(2000):
    shuffled_labels = shuffle(node_labels.copy())
    perm_intra, perm_inter = [], []
    for i in range(len(shuffled_labels)):
        for j in range(i + 1, len(shuffled_labels)):
            value = G[i][j][weight_type]
            if shuffled_labels[i] == shuffled_labels[j]:
                perm_intra.append(value)
            else:
                perm_inter.append(value)
    perm_diffs.append(np.mean(perm_intra) - np.mean(perm_inter))

perm_p = np.sum(np.array(perm_diffs) >= observed_diff) / len(perm_diffs)
print(f"Permutation Test p-value: {perm_p:.4f}")
if perm_p < 0.05:
    print("Reliable: Observed difference is significant vs. random label assignments.")


## 5. Cliff's Delta (effect size)

A non-parametric effect size for how much larger intra-category similarities tend to be than inter-category ones — complements the p-value from the Mann-Whitney test above with a magnitude.

In [ ]:
import numpy as np

weight_type = 'original_sim'

node_labels = [topic['model_label'] for topic in all_subtopics]

nodes = list(G.nodes())
if len(nodes) != len(all_subtopics):
    raise ValueError("Number of nodes in G must match length of all_subtopics")

intra_sims, inter_sims = [], []
for i, node_i in enumerate(nodes):
    for j, node_j in enumerate(nodes[i + 1:], start=i + 1):
        sim = G[node_i][node_j].get(weight_type, 0.0) if G.has_edge(node_i, node_j) else 0.0
        if node_labels[i] == node_labels[j]:
            intra_sims.append(sim)
        else:
            inter_sims.append(sim)


def cliffs_delta(x, y):
    """Vectorized Cliff's delta (avoids an O(n*m) Python double loop)."""
    x, y = np.array(x), np.array(y)
    gt = np.sum(x[:, np.newaxis] > y[np.newaxis, :])
    lt = np.sum(x[:, np.newaxis] < y[np.newaxis, :])
    return (gt - lt) / (len(x) * len(y))


delta = cliffs_delta(intra_sims, inter_sims)
print(f"Cliff's Delta: {delta:.4f}")
if delta > 0.474:
    print("Large effect: Strong category-specific structure.")
elif delta > 0.33:
    print("Medium effect.")
elif delta > 0.147:
    print("Small effect.")
elif delta > 0:
    print("Negligible positive effect.")
else:
    print("No or negative effect.")


## 6. Graph modularity and comparison to null models (structure reliability)

Filters the graph to a high-similarity threshold, computes modularity using the known categories as communities (under a few different edge-weight definitions), then compares against a configuration-model null to check the community structure isn't just an artifact of the degree sequence.

In [ ]:
import networkx as nx
from networkx.algorithms.community import modularity
from collections import defaultdict
import numpy as np
import pickle
import os

n = len(all_subtopics)
similarities = [sim_matrix[i, j] for i in range(n) for j in range(i + 1, n)]
similarities = np.array(similarities)

median = np.median(similarities)
mean = np.mean(similarities)
std = np.std(similarities)
sim_threshold = 0.993

print(f"median: {median}")
print(f"mean: {mean}")
print(f"std: {std}")
print(f"sim_threshold: {sim_threshold}")
print("=" * 64)

from matplotlib.cm import get_cmap

cmap = get_cmap("tab10", len(labels))
label_color = {label: cmap(i) for i, label in enumerate(labels)}
for node in G.nodes:
    if 'color' not in G.nodes[node]:
        G.nodes[node]['color'] = label_color[G.nodes[node]['label']]

filtered_edges = [(u, v) for u, v, d in G.edges(data=True) if d['original_sim'] >= 0.994]
filtered_G = G.edge_subgraph(filtered_edges).copy()

edge_count = filtered_G.number_of_edges()
print("\nNetwork statistics:")
print(f"  Total nodes: {filtered_G.number_of_nodes()}")
print(f"  Total edges: {edge_count}")
print(f"  Edge density: {edge_count / (n * (n - 1) // 2):.4f}")
print("=" * 64)

# Modularity using node labels (category) as communities
communities = defaultdict(list)
for node, data in filtered_G.nodes(data=True):
    communities[data['label']].append(node)
community_list = list(communities.values())

print("higher than 0.3-0.7 is good for real networks")
print("Weighted Modularity:", modularity(filtered_G, community_list, weight="weight"))
print("Unweighted Modularity:", modularity(filtered_G, community_list, weight=None))
print("normalized weight Modularity:", modularity(filtered_G, community_list, weight="nor_weight"))
obs_mod = modularity(filtered_G, community_list, weight="original_sim")
print("original_sim_Weighted Modularity:", obs_mod)
print("=" * 64)

# Null model test (configuration model preserves the observed degree sequence)
if filtered_G.number_of_edges() > 0:
    print("\nNull model test:")
    degrees = [d for _, d in filtered_G.degree()]
    null_mods = []
    n_permutations = 1000

    for _ in range(n_permutations):
        try:
            null_G = nx.configuration_model(degrees)
            null_G = nx.Graph(null_G)  # Remove multi-edges/self-loops
            null_mod = modularity(null_G, community_list, weight="original_sim") if null_G.number_of_edges() > 0 else 0.0
            null_mods.append(null_mod)
        except Exception:
            null_mods.append(0.0)

    null_mean = np.mean(null_mods)
    null_p = np.sum(np.array(null_mods) >= obs_mod) / len(null_mods)

    print(f"  Null mean modularity: {null_mean:.4f}")
    print(f"  p-value: {null_p:.4f}")

    if null_p < 0.05:
        print("  Result: Significant community structure (p < 0.05)")
        print("  ✓ Mental health topic clustering is significantly better than random")
    else:
        print("  Result: No significant community structure (p >= 0.05)")
        print("  ✗ Mental health topic clustering is not significantly different from random")
else:
    print("\nNo edges in graph, cannot perform null model test.")


## 7. Assortativity mixing

Do subtopics preferentially connect to other subtopics in the same category (assortative), or is mixing random/disassortative? Computed under several edge-weight definitions.

In [ ]:
import networkx as nx
import numpy as np


def weighted_attribute_assortativity_coefficient(G, attribute, weight_attr=None):
    """Weighted attribute assortativity coefficient.

    If weight_attr is None, this is equivalent to networkx's built-in
    (unweighted) attribute_assortativity_coefficient.
    """
    attr_values = set(nx.get_node_attributes(G, attribute).values())
    mapping = {val: idx for idx, val in enumerate(sorted(attr_values))}
    num_classes = len(mapping)

    M = np.zeros((num_classes, num_classes))
    total_weight = 0.0

    for u, v, data in G.edges(data=True):
        if attribute in G.nodes[u] and attribute in G.nodes[v]:
            au = mapping[G.nodes[u][attribute]]
            av = mapping[G.nodes[v][attribute]]
            w = data.get(weight_attr, 1.0) if weight_attr else 1.0
            M[au, av] += w
            M[av, au] += w  # Treat as two directed edges
            total_weight += 2 * w

    if total_weight == 0:
        return 0.0

    e = M / total_weight
    a = np.sum(e, axis=1)
    tr_e = np.trace(e)
    sum_a2 = np.dot(a, a)

    return 0.0 if (1 - sum_a2) == 0 else (tr_e - sum_a2) / (1 - sum_a2)


weight_types = [None, 'weight', 'nor_weight', 'original_sim']

print("\nAssortativity Test for Different Weights:")
for wt in weight_types:
    if wt is None:
        assortativity_score = nx.attribute_assortativity_coefficient(filtered_G, "label")
        wt_name = "unweighted"
    else:
        assortativity_score = weighted_attribute_assortativity_coefficient(filtered_G, "label", weight_attr=wt)
        wt_name = wt
    print(f"  Weight: {wt_name}")
    print(f"    Assortativity coefficient (r): {assortativity_score:.4f}")
    if assortativity_score > 0.3:
        print("    ✓ Strong assortative mixing (nodes prefer same community)")
    elif assortativity_score > 0:
        print("    ✓ Weak assortative mixing")
    elif assortativity_score == 0:
        print("    ~ No assortativity (random connections)")
    else:
        print("    ✗ Disassortative mixing (nodes prefer cross-community edges)")
    print("=" * 66)


## 8. Edge-level paired coherence

Per-edge c_v coherence between each connected pair of subtopics, for inspection alongside their similarity score.

In [ ]:
try:
    from gensim.models.coherencemodel import CoherenceModel
    from gensim.corpora.dictionary import Dictionary
except ImportError:
    print("gensim not available. Install it to use this metric.")
else:
    all_keywords = [[kw for kw, _ in t['keywords'][:10]] for t in all_subtopics]
    dictionary = Dictionary(all_keywords)

    def paired_coherence(topic1_keywords, topic2_keywords, top_n=10):
        combined = list(set(
            [kw for kw, _ in topic1_keywords[:top_n]] + [kw for kw, _ in topic2_keywords[:top_n]]
        ))
        if len(combined) < 2:
            return 0.0
        cm = CoherenceModel(topics=[combined], texts=all_keywords, dictionary=dictionary, coherence='c_v')
        return cm.get_coherence()

    edge_coherences = {}
    for u, v in filtered_G.edges:
        topic_u, topic_v = all_subtopics[u], all_subtopics[v]
        coherence = paired_coherence(topic_u['keywords'], topic_v['keywords'])
        edge_coherences[(u, v)] = coherence
        print(f"Edge ({topic_u['name']}, {topic_v['name']}) - Similarity: {filtered_G[u][v]['original_sim']:.4f}, Paired Coherence: {coherence:.4f}")

    if edge_coherences:
        avg_coherence = np.mean(list(edge_coherences.values()))
        print(f"Average Paired Coherence for connected edges: {avg_coherence:.4f}")


## 9. Community detection comparison

Compares the predefined categories against communities found automatically by greedy modularity maximization — do the categories used in labeling line up with what a community-detection algorithm would find on its own?

In [ ]:
import networkx as nx
from networkx.algorithms.community import modularity, greedy_modularity_communities
from collections import defaultdict

weight_type = "original_sim"

predefined_communities = defaultdict(list)
for node, data in filtered_G.nodes(data=True):
    predefined_communities[data['label']].append(node)

new_communities = list(greedy_modularity_communities(filtered_G, weight=weight_type))
print(f"\nDetected {len(new_communities)} communities.")

node_to_new_comm = {node: i for i, comm in enumerate(new_communities) for node in comm}

print("\nNew Communities (from Greedy Modularity):")
for i, comm in enumerate(new_communities):
    print(f"Community {i}:")
    for node in comm:
        print(f"  - {filtered_G.nodes[node]['name']}")

print("\nNode Community Assignments (Predefined vs. New):")
print(f"{'Subtopic':<30} {'Predefined Community':<25} {'New Community':<40}")
print("-" * 95)
for node in filtered_G.nodes:
    topic = filtered_G.nodes[node]['name']
    predefined_label = filtered_G.nodes[node]['label']
    new_comm_id = node_to_new_comm[node]
    print(f"{topic:<30} {predefined_label:<25} Community {new_comm_id:<40}")

mod_score = modularity(filtered_G, new_communities, weight=weight_type)
print(f"\nGraph Modularity (New Communities): {mod_score:.4f}")

predefined_comm_list = list(predefined_communities.values())
obs_mod = modularity(filtered_G, predefined_comm_list, weight=weight_type)
print(f"Observed Modularity (Predefined Communities): {obs_mod:.4f}")


## 10. Posts per year, by main category

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

current_dir = os.getcwd()
platform = "beyondblue"
labels = ["Anxiety", "Depression", "PTSD and trauma", "Suicidal thoughts and self-harm"]

main_symptoms_df = []
for label in labels:
    df = pd.read_csv(os.path.join(current_dir, 'data', f'{platform}_data', 'berttopic_label', label, 'label_df.csv'))
    main_symptoms_df.append(df)

main_symptoms_df = pd.concat(main_symptoms_df, ignore_index=True)
main_symptoms_df['year'] = pd.to_datetime(main_symptoms_df['Post Date']).dt.year
main_symptoms_df['Post Category'] = main_symptoms_df['Post Category'].str.strip()

print(main_symptoms_df.shape)
print("Missing values in 'year' column:", main_symptoms_df['year'].isna().sum())

post_counts = main_symptoms_df.groupby(['year', 'Post Category']).size().reset_index(name='Post Count')

plt.figure(figsize=(12, 6))
sns.lineplot(data=post_counts, x='year', y='Post Count', hue='Post Category', marker='o')
plt.xlabel('Year')
plt.ylabel('Number of Posts')
plt.xticks(rotation=45)
plt.legend(title='Post Category', loc='upper left')
plt.tight_layout()
plt.savefig("./pic/num_pos_by_yearcate.svg", bbox_inches="tight")
plt.show()


## 11. Posts per year, by subtopic within each category

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

current_dir = os.getcwd()
platform = "beyondblue"
labels = ["Anxiety", "Depression", "PTSD and trauma", "Suicidal thoughts and self-harm"]

for label in labels:
    df = pd.read_csv(os.path.join(current_dir, 'data', f'{platform}_data', 'berttopic_label', label, f'{label}_topic_docs.csv'))

    df['year'] = pd.to_datetime(df['Post Date']).dt.year
    df = df[df['Topic'] != -1]  # Exclude outliers
    topic_counts = df.groupby(['year', 'Topic']).size().reset_index(name='Post Count')
    topic_counts['Topic'] = topic_counts['Topic'].replace(custom_names_dict[label])

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=topic_counts, x='year', y='Post Count', hue='Topic', marker='o')
    plt.title(f'Number of Posts by Year and Topic for {label}')
    plt.xlabel('Year')
    plt.ylabel('Number of Posts')
    plt.xticks(rotation=45)
    plt.legend(title='Topic', loc='upper left')
    plt.tight_layout()
    plt.savefig(
        os.path.join(current_dir, 'data', f'{platform}_data', 'berttopic_label', label, f'{label}_topic_yearly_counts.svg'),
        bbox_inches="tight",
    )
    plt.show()
